## Setup & Configuration

### Imports
- **`sqlite3`** – Connect to and query the local SQLite database.
- **`pandas`** – Load query results into DataFrames for analysis and manipulation.
- **`numpy`** – Support for numerical operations and array handling.
- **`datetime`** – Work with date and time values (e.g., filtering or timestamping).

### Pandas Display Options
- `display.max_columns = 50` → Show up to 50 columns without truncation.
- `display.width = 200` → Widen the console output to prevent line wrapping.
- `display.float_format` → Format floats with thousands separators and 2 decimal places for readability.

### Database Configuration
- **`DB_PATH`** – Path to the SQLite database file (`vault_chain_flow.db`) located in the `customer_data` folder.
- **`TABLE`** – Name of the target table (`Customer_Transactions`) to query.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DB_PATH = r"../customer_data/vault_chain_flow.db"
TABLE = "Customer_Transactions"

### Connect to Database & List Tables

- **`con = sqlite3.connect(DB_PATH)`** – Open a connection to the SQLite database at the configured path.
- **`cursor = con.cursor()`** – Create a cursor object to execute SQL statements.
- **`cursor.execute(...)`** – Run a query against `sqlite_master` (SQLite's internal schema table) to retrieve the names of all tables of type `'table'` in the database.
- **`cursor.fetchall()`** – Fetch all matching rows; prints a list of table names so we can verify the database contents before querying.

In [2]:
con = sqlite3.connect(DB_PATH)
cursor = con.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print("Tables in database:", cursor.fetchall())

Tables in database: [('Customer_Transactions',)]


In [3]:
df = pd.read_sql_query(f"SELECT * FROM {TABLE}", con)
print(f"Loaded {len(df):,} rows and {df.shape[1]} columns.")

Loaded 580,962 rows and 14 columns.


### Missing Values per Column

Quick one-liner that returns the total count of `NaN` values for every column in `df`.

- Useful as a fast, inline check of data completeness — especially right after parsing dates (e.g., `pd.to_datetime` silently converts unparseable values to `NaT`, which show up here as missing).
- Compare against the earlier `missing` DataFrame to confirm nothing new was introduced by recent transformations.

In [5]:
df.head(10)

,TransactionID,CustomerID,TransactionDate,TransactionTime,TransactionType,TransactionAmount,TransactionCurrency,AssetType,CustomerDOB,CustGender,CustLocation,CustAccountBalance,WalletActivity,TransactionStatus
0,TXN00000001,CUST00001,2024-12-13,10:03:23,P2P Transfer,79.36,GBP,Traditional,2007-04-30,Male,London,"4,716.01",0,Completed
1,TXN00000002,CUST00001,2024-12-13,17:15:53,Bill Payment,54.95,GBP,Traditional,2007-04-30,Male,London,"4,661.06",0,Completed
2,TXN00000003,CUST00001,2024-12-14,12:26:16,Wallet Transfer,410.02,GBP,Traditional,2007-04-30,Male,None,"5,071.08",1,Completed
3,TXN00000004,CUST00001,2024-12-14,13:10:09,Withdrawal,212.52,USD,Traditional,2007-04-30,Male,London,"4,905.31",0,Completed
4,TXN00000005,CUST00001,2024-12-14,20:13:04,Bill Payment,674.77,GBP,Traditional,2007-04-30,Male,London,"4,230.54",0,Completed
5,TXN00000006,CUST00001,2024-12-15,07:10:38,Crypto Buy,74.81,USDT,USDT,2007-04-30,Male,London,"4,172.19",1,Completed
6,TXN00000007,CUST00001,2024-12-15,11:41:13,Cross-Border Payment,441.29,EUR,Traditional,2007-04-30,Male,London,"3,797.09",0,Completed
7,TXN00000008,CUST00001,2024-12-15,19:27:06,Deposit,275.91,EUR,Traditional,2007-04-30,Male,London,"4,031.62",0,Completed
8,TXN00000009,CUST00001,2024-12-16,17:44:56,Deposit,169.85,USD,Traditional,2007-04-30,Male,London,"4,164.10",0,Completed
9,TXN00000010,CUST00001,2024-12-21,08:17:59,P2P Transfer,182.63,USD,None,2007-04-30,Male,London,"4,021.65",0,Completed


In [4]:
df.isna().sum()

TransactionID              0
CustomerID                 0
TransactionDate            0
TransactionTime            0
TransactionType            0
TransactionAmount          0
TransactionCurrency    11846
AssetType              14475
CustomerDOB                0
CustGender                 0
CustLocation           17196
CustAccountBalance         0
WalletActivity             0
TransactionStatus          0
dtype: int64

### Imputing Missing `TransactionCurrency` with a Rule-Based Cascade

`TransactionCurrency` is a critical field — every transaction should have a currency for aggregation, FX conversion, and reconciliation. Rather than dropping rows or applying a naive global fill (e.g., mode), we apply a **prioritized cascade of 5 rules**, ordered from most-specific and reliable to most-generic fallback. Each rule targets only the rows still missing a currency, so earlier, higher-confidence rules always take precedence.

A shared lookup table maps each `AssetType` to its natural settlement currency: BTC → BTC, ETH → ETH, USDT → USDT, and Stablecoin → USDT (since stablecoins settle in USDT).

---

#### Rule 1: AssetType drives currency
**Logic:** If `TransactionCurrency` is missing **and** the row has a recognized `AssetType`, infer the currency directly from the asset.

- **Why first?** Asset type is the strongest signal — a BTC transaction is almost certainly denominated in BTC.
- The mask ensures we only touch rows that are *both* missing currency *and* have a known asset.
- Prints how many rows this rule filled.

---

#### Rule 2: Crypto transaction type → currency from asset
**Logic:** Crypto Buy/Sell transactions with a known asset inherit that asset's currency — even if Rule 1 didn't already catch them.

- **Why here?** Reinforces the crypto → asset currency assumption using `TransactionType` as an additional qualifier.
- Requires `AssetType` to be present so the lookup produces a valid value.

---

#### Rule 3: Wallet-only movements default to GBP
**Logic:** Rows flagged as wallet activity with no currency are internal wallet movements, which default to the home currency (GBP).

- **Why GBP?** Internal wallet activity doesn't cross borders, so it settles in the platform's home currency.

---

#### Rule 4: Cross-border defaults to USD
**Logic:** Cross-Border Payment transactions without a currency default to USD — the de facto standard for international settlement.

---

#### Rule 5: Everything else → GBP (home currency)
**Logic:** Catch-all fallback — any remaining rows missing a currency are filled with GBP.

- **Why last?** This is the lowest-confidence rule and should only apply after all specific rules have had a chance to fire. It guarantees zero remaining nulls.

---

#### Final Verification
Confirms the cascade fully resolved the column (expected: **0** remaining nulls).

---

### Key Design Principles

| Principle | Why It Matters |
|---|---|
| **Ordered specificity** | Rules run from most-specific (asset → currency) to most-generic (fallback → GBP), so high-confidence inferences always win. |
| **Null-guarded masks** | Every rule only applies where the currency is still missing, so no existing value is ever overwritten. |
| **Observability** | Each rule prints how many rows it filled — you can audit which rules did the heavy lifting and detect unexpected shifts in data distribution. |
| **Idempotent & reproducible** | Rules are deterministic; re-running the block on an already-filled column is a no-op. |

In [ ]:
# Rule 1 — AssetType drives currency
asset_to_currency = {
    "BTC": "BTC",
    "ETH": "ETH",
    "USDT": "USDT",
    "Stablecoin": "USDT",
}
mask = df["TransactionCurrency"].isna() & df["AssetType"].isin(asset_to_currency)
df.loc[mask, "TransactionCurrency"] = df.loc[mask, "AssetType"].map(asset_to_currency)
print(f"Rule 1 filled: {mask.sum():,}")

# Rule 2 — Wallet-only movements default to GBP
mask = df["TransactionCurrency"].isna() & (df["WalletActivity"] == 1)
df.loc[mask, "TransactionCurrency"] = "GBP"
print(f"Rule 2 filled: {mask.sum():,}")

# Rule 3 — Cross-border defaults to USD
mask = df["TransactionCurrency"].isna() & (df["TransactionType"] == "Cross-Border Payment")
df.loc[mask, "TransactionCurrency"] = "USD"
print(f"Rule 3 filled: {mask.sum():,}")

# Rule 4 — Everything else → GBP (home currency)
mask = df["TransactionCurrency"].isna()
df.loc[mask, "TransactionCurrency"] = "GBP"
print(f"Rule 4 filled: {mask.sum():,}")

print("\nRemaining TransactionCurrency nulls:", df["TransactionCurrency"].isna().sum())

Rule 1 filled: 3,217
Rule 3 filled: 1,835
Rule 4 filled: 559
Rule 5 filled: 6,235

Remaining TransactionCurrency nulls: 0


### Imputing Missing `AssetType` from `TransactionCurrency`

Now that `TransactionCurrency` is fully populated (from the previous cascade), we can reverse-infer `AssetType` for rows where it's missing. Because currency and asset type are tightly coupled — fiat currencies map to traditional assets, stablecoins to stablecoin assets, and crypto currencies to their matching crypto asset — this is a reliable, deterministic fill.

A shared reverse-lookup table maps each currency to its corresponding asset class:

- **GBP, EUR, USD** → Traditional
- **USDT** → Stablecoin
- **BTC** → BTC
- **ETH** → ETH

---

#### Step 1 — Fill `AssetType` from `TransactionCurrency`

**Logic:** For every row where `AssetType` is missing, look up the asset class based on the row's `TransactionCurrency` and assign it.

- **Why this works:** The previous step guaranteed every row has a currency, so the lookup can resolve almost every remaining null.
- A mask captures rows where `AssetType` is null **before** the fill, so we can report exactly how many rows were updated.
- Prints the count of rows filled.

---

#### Step 2 — Report remaining nulls

After the fill, print how many `AssetType` values are still missing. In theory this should be **0**, since every currency in the lookup maps to an asset — but it's good practice to verify rather than assume.

---

#### Step 3 — Safety net fallback

Any row still missing `AssetType` (e.g., if a currency existed that wasn't in the lookup table) is filled with **"Traditional"** using a simple `fillna`.

- **Why "Traditional"?** It's the most common asset class in the dataset and a conservative default — treating unknown rows as traditional fiat-backed activity rather than crypto.
- **Why this step exists:** Defensive programming — it guarantees the column has zero nulls downstream, so no later analysis or model breaks on missing values.

---

### Why This Order Matters

| Step | Purpose |
|---|---|
| **Reverse-infer from currency** | Leverages the now-complete currency column to reconstruct asset type with high confidence. |
| **Verify remaining nulls** | Confirms the mapping covered all observed currencies; surfaces surprises for investigation. |
| **Safety net fill** | Guarantees no nulls survive, protecting downstream operations (grouping, modeling, encoding). |

Together with the earlier currency imputation, this makes the `TransactionCurrency` ↔ `AssetType` pair **mutually consistent and fully populated** — a solid foundation for any asset-level or currency-level aggregation that follows.

In [7]:
currency_to_asset = {
    "GBP": "Traditional",
    "EUR": "Traditional",
    "USD": "Traditional",
    "USDT": "Stablecoin",
    "BTC": "BTC",
    "ETH": "ETH",
}

mask = df["AssetType"].isna()
df.loc[mask, "AssetType"] = df.loc[mask, "TransactionCurrency"].map(currency_to_asset)

print(f"AssetType filled from currency: {mask.sum():,}")
print("Remaining AssetType nulls:", df["AssetType"].isna().sum())

# Safety net — anything still null (shouldn't be) → Traditional
df["AssetType"] = df["AssetType"].fillna("Traditional")

AssetType filled from currency: 14,475
Remaining AssetType nulls: 0


### Imputing Missing `CustLocation` via Customer Mode + Fallback

Customer location is often missing on individual transactions but can be reliably inferred from the customer's *other* transactions — most customers operate from a consistent location. This step applies a two-tier imputation:

---

#### Step 1 — Compute each customer's modal location

For every `CustomerID`, calculate the most frequently occurring non-null `CustLocation` across all of that customer's transactions.

- Uses `groupby("CustomerID").transform(...)` so the result is a Series **aligned to the original DataFrame index** — no merge or temp column needed.
- The lambda drops nulls first, so missing values don't skew the mode.
- If a customer has **no** recorded location at all, the result is `NaN` (handled in Step 2).

---

#### Step 2 — Fill nulls with the customer's mode, then fall back to "Unknown"

Two chained `fillna` calls resolve the column in priority order:

1. **Fill from the customer's modal location** — high-confidence inference for customers who have at least one known location elsewhere.
2. **Fall back to `"Unknown"`** — for customers with *zero* known locations, we assign a sentinel value so the column has no nulls downstream.

The chained pattern reads top-to-bottom as: *"use the mode if we have one, otherwise mark it unknown."*

---

#### Step 3 — Verify

Prints the remaining null count in `CustLocation` — expected to be **0**, confirming the column is fully populated.

---



In [8]:
mode_per_customer = df.groupby("CustomerID")["CustLocation"].transform(
    lambda s: s.dropna().mode().iloc[0] if s.notna().any() else np.nan
)

df["CustLocation"] = df["CustLocation"].fillna(mode_per_customer).fillna("Unknown")

print("Remaining CustLocation nulls:", df["CustLocation"].isna().sum())

Remaining CustLocation nulls: 0


In [9]:
df["CustLocation"].value_counts()

CustLocation
London                  122409
Manchester               46714
Birmingham               43160
Leeds                    35836
Glasgow                  30496
Bristol                  29452
Liverpool                28112
Edinburgh                25299
Sheffield                21724
Belfast                  20468
Newcastle                18798
Leicester                18361
Cardiff                  17852
Brighton                 17841
Southampton              17704
Nottingham               16845
International - EU       12046
Milton Keynes            11890
Coventry                 11178
Reading                  10281
Aberdeen                  9601
International - US        7955
International - APAC      6936
Unknown                      4
Name: count, dtype: int64

In [10]:
df.isna().sum()

TransactionID          0
CustomerID             0
TransactionDate        0
TransactionTime        0
TransactionType        0
TransactionAmount      0
TransactionCurrency    0
AssetType              0
CustomerDOB            0
CustGender             0
CustLocation           0
CustAccountBalance     0
WalletActivity         0
TransactionStatus      0
dtype: int64

In [11]:
df.head(10)

,TransactionID,CustomerID,TransactionDate,TransactionTime,TransactionType,TransactionAmount,TransactionCurrency,AssetType,CustomerDOB,CustGender,CustLocation,CustAccountBalance,WalletActivity,TransactionStatus
0,TXN00000001,CUST00001,2024-12-13,10:03:23,P2P Transfer,79.36,GBP,Traditional,2007-04-30,Male,London,"4,716.01",0,Completed
1,TXN00000002,CUST00001,2024-12-13,17:15:53,Bill Payment,54.95,GBP,Traditional,2007-04-30,Male,London,"4,661.06",0,Completed
2,TXN00000003,CUST00001,2024-12-14,12:26:16,Wallet Transfer,410.02,GBP,Traditional,2007-04-30,Male,London,"5,071.08",1,Completed
3,TXN00000004,CUST00001,2024-12-14,13:10:09,Withdrawal,212.52,USD,Traditional,2007-04-30,Male,London,"4,905.31",0,Completed
4,TXN00000005,CUST00001,2024-12-14,20:13:04,Bill Payment,674.77,GBP,Traditional,2007-04-30,Male,London,"4,230.54",0,Completed
5,TXN00000006,CUST00001,2024-12-15,07:10:38,Crypto Buy,74.81,USDT,USDT,2007-04-30,Male,London,"4,172.19",1,Completed
6,TXN00000007,CUST00001,2024-12-15,11:41:13,Cross-Border Payment,441.29,EUR,Traditional,2007-04-30,Male,London,"3,797.09",0,Completed
7,TXN00000008,CUST00001,2024-12-15,19:27:06,Deposit,275.91,EUR,Traditional,2007-04-30,Male,London,"4,031.62",0,Completed
8,TXN00000009,CUST00001,2024-12-16,17:44:56,Deposit,169.85,USD,Traditional,2007-04-30,Male,London,"4,164.10",0,Completed
9,TXN00000010,CUST00001,2024-12-21,08:17:59,P2P Transfer,182.63,USD,Traditional,2007-04-30,Male,London,"4,021.65",0,Completed


In [12]:
df.duplicated().sum()
df['TransactionID'].duplicated().sum()

0

In [13]:
import os
os.makedirs("../customer_data/artifacts/", exist_ok=True)

In [14]:
df.to_csv("../customer_data/artifacts/cleaned_transaction_data.csv", index=False)